# KKBox Churn Prediction — EDA & Feature Engineering

Builds on `master_merged.csv` from `01_data_foundation_preprocessing.ipynb`
(25,000 sampled users, 25 columns, ~9% churn). This notebook explores the
data against the churn target, engineers new features, and writes
`master_features.csv` as the handoff artifact for modeling.

**Update `DATA_DIR` / `OUTPUT_DIR` below to your own local path before running.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

DATA_DIR = r"C:\Users\HP\Desktop\KKbox_churn"
OUTPUT_DIR = r"C:\Users\HP\Desktop\KKbox_churn"

CHURN_CUTOFF = pd.Timestamp("2017-03-31")  # label decision date, per source notebook

## 1. Load & sanity-check

Confirm shape and churn rate match the source notebook, and re-cast date
columns to datetime (CSV round-trips often turn them back into strings).

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/master_merged.csv")
print("shape:", df.shape)
df.head()

In [ ]:
date_cols = [
    "registration_init_time",
    "last_transaction_date",
    "membership_expire_date",
    "first_log_date",
    "last_log_date",
]
date_cols = [c for c in date_cols if c in df.columns]  # only keep columns that actually exist

for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

print(df[date_cols].dtypes)
print()
print(df.columns.tolist())

In [ ]:
churn_rate = df["is_churn"].mean()
print(f"churn rate: {churn_rate:.1%}")
assert df.shape == (25000, 25), "shape doesn't match source notebook — check the sample wasn't corrupted"
df["is_churn"].value_counts(normalize=True)

## 2. Univariate EDA

### Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
df["is_churn"].value_counts().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_xticklabels(["Retained", "Churned"], rotation=0)
ax.set_title("Churn distribution")
plt.tight_layout()
plt.show()

### Numeric feature distributions

In [ ]:
numeric_cols = [
    "bd_clean",
    "txn_count",
    "total_secs_sum",
    "num_unq_mean",
    "active_days",
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4 * len(numeric_cols), 4))
for ax, col in zip(axes, numeric_cols):
    df[col].dropna().hist(ax=ax, bins=40)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/../../churn_distributions.png", dpi=150)
plt.show()

Flag any heavily right-skewed columns (e.g. `total_secs_sum`) as
log-transform candidates before modeling.

### Categorical frequency counts

In [ ]:
categorical_cols = ["city", "gender", "registered_via", "last_payment_method"]
categorical_cols = [c for c in categorical_cols if c in df.columns]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))

### Missingness / flag rates

In [ ]:
flag_cols = ["has_member_record", "has_transactions", "has_logs", "is_age_missing"]
flag_cols = [c for c in flag_cols if c in df.columns]
df[flag_cols].mean().rename("rate=True").to_frame()

## 3. Bivariate EDA (vs. churn)

This is the core of the notebook — for each feature, does it separate
churners from non-churners?

### Categorical vs. churn — churn rate per category

In [ ]:
for col in categorical_cols:
    rate = df.groupby(col)["is_churn"].mean().sort_values(ascending=False)
    print(f"\n--- churn rate by {col} (top 10) ---")
    print(rate.head(10))

### Key boolean flags vs. churn

In [ ]:
flag_vs_churn = {}
for col in ["has_transactions", "has_logs", "is_auto_renew", "any_cancel"]:
    if col in df.columns:
        flag_vs_churn[col] = df.groupby(col)["is_churn"].mean()

flag_summary = pd.DataFrame(flag_vs_churn)
flag_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
flag_summary.T.plot(kind="bar", ax=ax)
ax.set_ylabel("churn rate")
ax.set_title("Churn rate by key flags")
ax.legend(title="flag value")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/../../churn_rate_flags.png", dpi=150)
plt.show()

### Numeric vs. churn — grouped distributions

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4 * len(numeric_cols), 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df, x="is_churn", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### Correlation heatmap (numeric features)

In [ ]:
corr_cols = [c for c in df.columns if df[c].dtype in ("int64", "float64") and c != "is_churn"]
corr = df[corr_cols + ["is_churn"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation heatmap")
plt.tight_layout()
plt.show()

**Note:** aggregate columns like `*_count` and `*_sum` will likely be
collinear — expected, keep in mind for feature selection at modeling time.

## 4. Feature engineering

Building on what's already aggregated in `master_merged.csv`.

### Recency & tenure

In [ ]:
df["days_since_last_transaction"] = (CHURN_CUTOFF - df["last_transaction_date"]).dt.days
df["days_since_last_log"] = (CHURN_CUTOFF - df["last_log_date"]).dt.days

if "registration_init_time" in df.columns:
    df["tenure_days"] = (CHURN_CUTOFF - df["registration_init_time"]).dt.days

df[["days_since_last_transaction", "days_since_last_log", "tenure_days"]].describe()

In [ ]:
# sanity check the recency cliff noted in EDA_SUMMARY.md
recency_bins = pd.cut(df["days_since_last_transaction"], bins=[-1, 7, 14, 21, 28, 35, 10000])
df.groupby(recency_bins)["is_churn"].mean()

### Engagement ratios

In [ ]:
if {"num_100_sum", "num_25_sum"}.issubset(df.columns):
    df["completion_ratio"] = df["num_100_sum"] / (df["num_100_sum"] + df["num_25_sum"]).replace(0, np.nan)

if {"total_secs_sum", "active_days"}.issubset(df.columns):
    df["mean_secs_per_active_day"] = df["total_secs_sum"] / df["active_days"].replace(0, np.nan)

### Price / value signals

In [ ]:
if {"last_amount_paid", "last_plan_price"}.issubset(df.columns):
    df["is_discounted"] = (df["last_amount_paid"] < df["last_plan_price"]).astype(int)

### Interaction / at-risk flags

In [ ]:
if {"is_auto_renew", "any_cancel"}.issubset(df.columns):
    df["at_risk_flag"] = ((df["is_auto_renew"] == 0) & (df["any_cancel"] == 1)).astype(int)

df["at_risk_flag"].value_counts() if "at_risk_flag" in df.columns else None

### Encode categoricals

Raw categorical columns are kept as-is (per the source notebook's note that
encoding is deferred to modeling); one-hot versions are added alongside so
the modeler can choose either.

In [ ]:
encode_cols = [c for c in ["city", "gender", "registered_via", "last_payment_method"] if c in df.columns]
df = pd.get_dummies(df, columns=encode_cols, prefix=encode_cols, dummy_na=True, drop_first=False)

print("new shape after encoding:", df.shape)

## 5. Output

Save the engineered feature table as the handoff artifact for modeling.
All cell outputs should be cleared before committing, per repo convention
(run top-to-bottom on handoff).

In [ ]:
out_path = f"{OUTPUT_DIR}/master_features.csv"
df.to_csv(out_path, index=False)
print("saved:", out_path, df.shape)

### Data dictionary — new engineered columns

| Column | Meaning |
|---|---|
| `days_since_last_transaction` | Days between last transaction and the 2017-03-31 churn cutoff |
| `days_since_last_log` | Days between last activity log and the churn cutoff |
| `tenure_days` | Days since registration, relative to the churn cutoff |
| `completion_ratio` | `num_100_sum / (num_100_sum + num_25_sum)` — fraction of full vs. short plays |
| `mean_secs_per_active_day` | `total_secs_sum / active_days` |
| `is_discounted` | 1 if last amount paid was below the plan price |
| `at_risk_flag` | 1 if auto-renew is off **and** the user has any cancellation on record |
| `{col}_*` one-hot columns | One-hot encoded versions of `city`, `gender`, `registered_via`, `last_payment_method` |

Full findings and modeling recommendations are written up separately in
`EDA_SUMMARY.md`.